In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [4]:
class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom Scikit-Learn Transformer that learns group statistics ONLY on the
    training fold during fit(), preventing data leakage during cross-validation.
    """
    def __init__(self):
        self.age_medians_ = None
        self.global_age_median_ = None
        self.fare_medians_ = None
        self.ticket_counts_ = None

    def _extract_title(self, series: pd.Series) -> pd.Series:
        titles = series.str.extract(r' ([A-Za-z]+)\.', expand=False)
        title_mapping = {
            'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
            'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
            'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
            'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
        }
        return titles.replace(title_mapping).fillna('Rare')

    def fit(self, X: pd.DataFrame, y=None):
        df = X.copy()
        df['Title'] = self._extract_title(df['Name'])
        
        # Learn hierarchical Age medians (Title + Pclass) strictly from training fold
        self.age_medians_ = df.groupby(['Title', 'Pclass'])['Age'].median().to_dict()
        self.global_age_median_ = df['Age'].median()
        
        # Learn Fare medians by Pclass
        self.fare_medians_ = df.groupby('Pclass')['Fare'].median().to_dict()
        
        # Learn Ticket group frequencies
        self.ticket_counts_ = df['Ticket'].value_counts().to_dict()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()
        
        # 1. Title extraction
        df['Title'] = self._extract_title(df['Name'])
        
        # 2. Contextual Age Imputation
        def impute_age(row):
            if pd.notnull(row['Age']):
                return row['Age']
            return self.age_medians_.get(
                (row['Title'], row['Pclass']), self.global_age_median_
            )
        
        df['Age_Imputed'] = df.apply(impute_age, axis=1)
        
        # 3. Family Size & Categorization
        df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
        df['FamilyBin'] = pd.cut(
            df['FamilySize'],
            bins=[0, 1, 4, 20],
            labels=['Singleton', 'SmallFamily', 'LargeFamily']
        ).astype(str)
        
        # 4. Ticket Group & Adjusted Fare Per Person
        # Map learned counts; default to 1 for unseen tickets in test set
        ticket_group = df['Ticket'].map(self.ticket_counts_).fillna(1)
        df['GroupSize'] = np.maximum(df['FamilySize'], ticket_group)
        
        # Impute missing Fare using Pclass median before dividing
        df['Fare_Imputed'] = df.apply(
            lambda r: self.fare_medians_.get(r['Pclass'], 14.45) if pd.isnull(r['Fare']) else r['Fare'],
            axis=1
        )
        df['FarePerPerson'] = df['Fare_Imputed'] / df['GroupSize']
        df['LogFarePerPerson'] = np.log1p(df['FarePerPerson'])
        
        # 5. Cabin & Deck proxy
        df['Deck'] = df['Cabin'].astype(str).str[0].replace('n', 'M') # 'n' from 'nan' -> 'M' (Missing)
        df['Deck'] = df['Deck'].replace({'T': 'M', 'G': 'FG', 'F': 'FG', 'A': 'ABC', 'B': 'ABC', 'C': 'ABC', 'D': 'DE', 'E': 'DE'})
        
        # 6. Interaction Term: Age * Class
        df['Age_Class_Interaction'] = df['Age_Imputed'] * df['Pclass']
        
        # Select engineered features to pass downstream
        feature_cols = [
            'Pclass', 'Sex', 'Embarked', 'Title', 'FamilyBin', 'Deck',
            'Age_Imputed', 'LogFarePerPerson', 'GroupSize', 'Age_Class_Interaction'
        ]
        return df[feature_cols]

In [5]:
def build_scalable_pipeline():
    categorical_cols = ['Pclass', 'Sex', 'Embarked', 'Title', 'FamilyBin', 'Deck']
    numerical_cols = ['Age_Imputed', 'LogFarePerPerson', 'GroupSize', 'Age_Class_Interaction']

    # Preprocessing branches
    cat_processor = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    num_processor = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()) # Crucial for Logistic Regression convergence
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('cat', cat_processor, categorical_cols),
        ('num', num_processor, numerical_cols)
    ])

    # Model Diversity: Linear + Bagging Trees + Boosted Trees
    # Note: Regularization (max_depth, min_samples_leaf, C) is strict to prevent memorizing 891 rows
    clf_logreg = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
    
    clf_rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_split=6,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    
    clf_gb = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        min_samples_leaf=5,
        random_state=42
    )

    # Soft Voting blends calibrated probabilities across orthogonal model architectures
    ensemble = VotingClassifier(
        estimators=[
            ('logreg', clf_logreg),
            ('rf', clf_rf),
            ('gb', clf_gb)
        ],
        voting='soft',
        weights=[1, 2, 2]
    )

    full_pipeline = Pipeline(steps=[
        ('feature_engineering', TitanicFeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', ensemble)
    ])

    return full_pipeline

In [6]:
if __name__ == "__main__":
    # Load Kaggle competition data (update paths as needed)
    train_df = pd.read_csv("/kaggle/input/datasets/ayushgauniyalhere/titanic-dataset/train.csv")
    test_df = pd.read_csv("/kaggle/input/datasets/ayushgauniyalhere/titanic-dataset/test.csv")

    X_train = train_df.drop(columns=['Survived', 'PassengerId'])
    y_train = train_df['Survived']
    X_test = test_df.drop(columns=['PassengerId'])

    pipeline = build_scalable_pipeline()

    # Rule #1 of Kaggle: Trust your Local Stratified CV, not the Public Leaderboard
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)

    print(f"10-Fold Stratified CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    # Fit on full training dataset and generate submission
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    submission = pd.DataFrame({
        'PassengerId': test_df['PassengerId'],
        'Survived': predictions
    })
    submission.to_csv("submission.csv", index=False)
    print("Saved leak-free predictions to submission.csv!")

10-Fold Stratified CV Accuracy: 0.8372 (+/- 0.0305)
Saved leak-free predictions to submission.csv!
